In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sqlalchemy import text

In [3]:
DB_USER = 'postgres'          
DB_PASSWORD = 'Neha2004*' 
DB_HOST = 'localhost'
DB_PORT = '5432'
DB_NAME = 'bluestock_mf'

engine = create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

In [17]:
fund_master= 'C:/Users/nehas/Downloads/01_fund_master_cleaned_1.csv'
nav= 'C:/Users/nehas/Downloads/02_nav_history_cleaned_1.csv'
aum= 'C:/Users/nehas/Downloads/03_aum_by_fund_house_cleaned_1.csv'
sip_inflows= 'C:/Users/nehas/Downloads/04_monthly_sip_inflows_cleaned_1.csv'
category_inflows= 'C:/Users/nehas/Downloads/05_category_inflows_cleaned_1.csv'
industry_folio= 'C:/Users/nehas/Downloads/06_industry_folio_count_cleaned_1.csv'
performance= 'C:/Users/nehas/Downloads/07_scheme_performance_cleaned_1.csv'
transactions= 'C:/Users/nehas/Downloads/08_investor_transactions_cleaned_1.csv'
portfolio= 'C:/Users/nehas/Downloads/09_portfolio_holdings_cleaned_1.csv'
benchmark= 'C:/Users/nehas/Downloads/10_benchmark_indices_cleaned_1.csv'




In [18]:
print("Loading dim_fund...")
fund_df = pd.read_csv(fund_master)
fund_df = fund_df[['amfi_code', 'scheme_name', 'fund_house', 'category', 
                   'risk_category']]

fund_df.to_sql('dim_fund', engine, if_exists='append', index=False)
print(f"dim_fund loaded: {len(fund_df)} rows")


Loading dim_fund...
dim_fund loaded: 40 rows


In [20]:
print("Loading dim_date...")
nav_df = pd.read_csv(nav)
nav_df['date'] = pd.to_datetime(nav_df['date'],errors='coerce')
date_dim = nav_df[['date']].drop_duplicates().sort_values('date').reset_index(drop=True)
date_dim['month'] = date_dim['date'].dt.month
date_dim['year'] = date_dim['date'].dt.year
date_dim['date_id'] = range(1, len(date_dim) + 1)
date_dim.to_sql('dim_date', engine, if_exists='append', index=False)

Loading dim_date...


150

In [21]:
def load_fact(table_name, csv_path, col_map, date_col='date', amfi_col='amfi_code'):
    df = pd.read_csv(csv_path)
    
    # Standardize keys
    if amfi_col in df.columns:
        df[amfi_col] = df[amfi_col].astype(str).str.strip()
    
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce').dt.normalize()
    
    with engine.connect() as conn:
        fund_map = pd.read_sql("SELECT fund_id, amfi_code FROM dim_fund", conn)
        fund_map['amfi_code'] = fund_map['amfi_code'].astype(str).str.strip()
        date_map = pd.read_sql("SELECT date_id, date FROM dim_date", conn)
        date_map['date'] = pd.to_datetime(date_map['date']).dt.normalize()
    
    # Merge fund
    df = df.merge(fund_map, left_on=amfi_col, right_on='amfi_code', how='left')
    
    # Merge date
    if date_col in df.columns:
        df = df.merge(date_map, left_on=date_col, right_on='date', how='left')
    
    # Select + rename
    select_cols = list(col_map.keys())
    df = df[select_cols].rename(columns=col_map)
    df = df.dropna(subset=['fund_id', 'date_id'])
    
    df.to_sql(table_name, engine, if_exists='append', index=False)
    print(f"{table_name} loaded: {len(df)} rows")



In [23]:
load_fact('fact_nav', nav, 
          {'fund_id': 'fund_id', 'date_id': 'date_id', 'nav': 'nav'})

fact_nav loaded: 46000 rows


In [25]:
load_fact('fact_transactions', transactions, 
          {'fund_id':'fund_id', 'date_id':'date_id', 'amount_inr':'amount_inr', 
           'transaction_type':'transaction_type', 'state':'state'},date_col='date')

fact_transactions loaded: 23547 rows


In [43]:
print("Loading fact_performance...")

# Read the CSV
performance = pd.read_csv(performance)

# Standardize amfi_code
performance['amfi_code'] = performance['amfi_code'].astype(str).str.strip()

# Load fund mapping
fund_map = pd.read_sql("""
SELECT fund_id, amfi_code
FROM dim_fund
""", engine)

fund_map['amfi_code'] = fund_map['amfi_code'].astype(str).str.strip()

# Merge
performance = performance.merge(
    fund_map,
    on="amfi_code",
    how="left"
)

# Keep required columns
performance = performance[
    [
        "fund_id",
        "return_1yr_pct",
        "return_3yr_pct",
        "return_5yr_pct",
        "expense_ratio_pct"
    ]
]

# Remove rows without matching fund_id
performance = performance.dropna(subset=["fund_id"])

# Load into PostgreSQL
performance.to_sql(
    "fact_performance",
    engine,
    if_exists="append",
    index=False
)

print(f"fact_performance loaded: {len(performance)} rows")

Loading fact_performance...
fact_performance loaded: 40 rows


In [52]:
aum = pd.read_csv(aum)

In [66]:
# Remove duplicate date_id columns
cols_to_drop = []

for col in ['date_id_x', 'date_id_y']:
    if col in aum.columns:
        cols_to_drop.append(col)

if cols_to_drop:
    aum = aum.drop(columns=cols_to_drop)

print(aum.columns.tolist())

['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes', 'date_id']


In [ ]:
fact_aum = aum[
    [
        'date_id',
        'fund_house',
        'aum_lakh_crore',
        'aum_crore',
        'num_schemes'
    ]
]

fact_aum.to_sql(
    'fact_aum',
    engine,
    if_exists='append',
    index=False
)

print(f"fact_aum loaded successfully: {len(fact_aum)} rows")

fact_aum loaded successfully
fact_aum loaded successfully: 90 rows
